# Multi-Asset Scenario Engine — v20 release notebook

A clean, package-backed reference workflow for **joint physical-measure scenario generation**, **1W/1Y horizon repricing**, and **portfolio VaR/Expected Shortfall analysis** across US equities, Treasury rates, credit spreads and FX.

The numerical building blocks live in `src/multi_asset_scenario_engine/`; this notebook is the reproducible research workflow rather than a second copy of the implementation.

**Core pipeline**

1. download and align equity/rates/credit/FX data;
2. build modular equity-premium and cross-asset factor blocks;
3. generate joint scenarios with block-bootstrap dynamics and optional GJR-GARCH FHS / OU level mean reversion;
4. reprice equities, Treasury and corporate bonds, and an FX forward at the horizon;
5. validate dependence, VaR and Expected Shortfall;
6. evaluate tail contributions and long-only minimum-CVaR allocations.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.offsets import BDay

from multi_asset_scenario_engine import *
from multi_asset_scenario_engine.config import *

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
plt.rcParams["figure.figsize"] = (10, 6)


## Configuration

The source v20 notebook kept conservative defaults for backward compatibility. For the public research release the switches below expose the mature architecture without forcing the expensive full-window backtest on every run. Set `RUN_FULL_VALIDATION = True` to activate the v20 full-history point-in-time VaR/ES evaluation.


In [ ]:
# Mature v20 scenario configuration
SAMPLE_MODE = "long"
CREDIT_BLOCK_SOURCE = "moody_2pt"
CONDITIONAL_VOL = "ewma_fhs"
VOL_PARAMS_MODE = "estimated_gjr"
LEVEL_MEAN_REVERSION = "ou"
BOOTSTRAP_BLOCK_MODE = "fixed"
DRIVER_DYNAMICS = "block_bootstrap"
CRISIS_MODE = "none"

# Keep the default notebook run practical; enable the full v20 validation explicitly.
RUN_FULL_VALIDATION = False
BACKTEST_MODE = "full" if RUN_FULL_VALIDATION else "recent"

# Horizon switch: use "1Y" to activate migration/default and annual carry conventions.
HORIZON_MODE = "1W"
HORIZON_BDAYS = 5 if HORIZON_MODE == "1W" else 252
HORIZON_YEARS = HORIZON_BDAYS / 252.0
CREDIT_MIGRATION_ENABLED = (HORIZON_MODE == "1Y")

print({
    "sample": SAMPLE_MODE, "credit": CREDIT_BLOCK_SOURCE, "vol": VOL_PARAMS_MODE,
    "level_mr": LEVEL_MEAN_REVERSION, "horizon": HORIZON_MODE, "backtest": BACKTEST_MODE
})



## 1. Load and align market data

The equity universe is now intentionally broader than the earlier tech-only prototype. The cross-asset input layer includes:

- a **multi-sector US equity sample**,
- a **market proxy**,
- **Treasury key rates**,
- and daily **observable factor data** used by the APT engine.


In [ ]:
# Download and align the data (equity, Treasury curve, Fama-French factors, and the credit curve)
# Per-block sample starts. SAMPLE_MODE="short" reproduces the single START_DATE window exactly;
# "long" lets each block bring its full history -- the aligned window is whatever the data jointly
# support, and the diagnostic below shows which block binds.
def _block_start(block):
    return BLOCK_START_DATES.get(block, START_DATE) if SAMPLE_MODE == "long" else START_DATE


stock_prices_raw, market_proxy_raw, rates_raw = download_market_data(
    tickers=TICKERS,
    market_proxy=MARKET_PROXY,
    rate_tickers=RATE_TICKERS,
    start_date=_block_start("equity"),
    download_end=DOWNLOAD_END,
)

ff_factor_panel_raw = download_fama_french_daily_factors(
    start_date=_block_start("equity"),
    end_date=DOWNLOAD_END,
)

# Credit block: ICE BofA OAS by rating bucket (see note in the config cell on the FRED ~3y window).
credit_spreads_raw = download_credit_oas(
    credit_oas_series=CREDIT_OAS_SERIES,
    start_date=_block_start("credit"),
    download_end=DOWNLOAD_END,
)

# FX block: currency-pair spot from FRED (long history; see config note). One pair (EUR/USD) here.
fx_rates_raw = download_fx_rates(
    fx_series=FX_SERIES,
    start_date=_block_start("fx"),
    download_end=DOWNLOAD_END,
)

# Crisis backbone: long-history Moody's Baa/Aaa yields (decades incl. every postwar crisis). Pulled
# from MOODYS_START_DATE (independent of the short main-sample window) to calibrate the crisis tail.
moodys_raw = download_moodys_credit(
    moodys_series=MOODYS_SERIES,
    start_date=MOODYS_START_DATE,
    download_end=DOWNLOAD_END,
)

# Credit DRIVER panel per CREDIT_BLOCK_SOURCE. "oas_buckets": the 7-bucket OAS panel drives the
# credit factors (v18 behaviour; FRED serves a rolling ~3y of it). "moody_2pt": a real, never-
# truncated 2-point curve -- Moody's seasoned Aaa/Baa yields minus the 10Y Treasury, both decimals,
# daily since the 1980s -- drives them instead; the bucket panel is kept aside as today's repricing
# ANCHOR and for the sensitivity BRIDGE (built in the joint-state section).
credit_bucket_panel = credit_spreads_raw.dropna(how="any")
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    _m10 = rates_raw["DGS10"].dropna()
    credit_driver_raw = pd.DataFrame({
        "Aaa_spr": moodys_raw["Aaa"] - _m10,
        "Baa_spr": moodys_raw["Baa"] - _m10,
    }).dropna(how="any")
else:
    credit_driver_raw = credit_spreads_raw

effective_date = choose_effective_asof(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    AS_OF_DATE,
    credit=credit_driver_raw,
    fx=fx_rates_raw,
)

stock_prices, market_proxy_prices, key_rates, ff_factor_panel, credit_spreads, fx_rates = align_market_data(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    effective_date,
    credit=credit_driver_raw,
    fx=fx_rates_raw,
)

# Long-window contiguity guard (LONG_GAP_TOLERANCE_DAYS in the config cell): a day-to-day change
# taken across a multi-year data hole -- the known case is the DGS30 discontinuation 2002-02 ->
# 2006-02 -- would enter the bootstrap pool as ONE fake extreme observation, poison for a tail
# engine. If the aligned window contains such a gap, trim it to start after the last one. The
# walk-forward backtest applies the same rule to every refitted window (same helper).
_win_start, _n_gaps, _max_gap = contiguous_window_start(stock_prices.index, LONG_GAP_TOLERANCE_DAYS)
if _win_start is not None:
    print(f"NOTE: aligned window contains {_n_gaps} internal gap(s) longer than "
          f"{LONG_GAP_TOLERANCE_DAYS} calendar days (largest: {_max_gap}d); trimming the "
          f"window start to {_win_start.date()} so no daily change spans a data hole.\n")
    stock_prices = stock_prices.loc[_win_start:]
    market_proxy_prices = market_proxy_prices.loc[_win_start:]
    key_rates = key_rates.loc[_win_start:]
    ff_factor_panel = ff_factor_panel.loc[_win_start:]
    credit_spreads = credit_spreads.loc[_win_start:]
    fx_rates = fx_rates.loc[_win_start:]

sector_table = pd.Series(SECTOR_MAP, name="sector").to_frame()

_credit_lbl = ("credit 2pt (Moody's - DGS10)" if CREDIT_BLOCK_SOURCE == "moody_2pt"
               else "credit OAS (all buckets)")
_first_usable = {
    "equity (all tickers)": stock_prices_raw.dropna(how="any").index.min(),
    "rates (all tenors)": rates_raw.dropna(how="any").index.min(),
    "FF factors": ff_factor_panel_raw.dropna(how="any").index.min(),
    _credit_lbl: credit_driver_raw.dropna(how="any").index.min(),
    "fx": fx_rates_raw.dropna(how="any").index.min(),
}
_binding = max(_first_usable, key=lambda kk: _first_usable[kk])
print(f"Sample mode               : {SAMPLE_MODE}   |   credit block source: {CREDIT_BLOCK_SOURCE}")
print("First usable date per block (all series present):")
for _k, _v in _first_usable.items():
    print(f"  {_k:28s}: {_v.date()}" + ("   <- binds the joint window" if _k == _binding else ""))
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    print(f"  {'bucket OAS (anchor/bridge)':28s}: {credit_bucket_panel.index.min().date()} "
          "(kept OUT of the joint alignment; anchors repricing and calibrates the bridge)")
if SAMPLE_MODE == "long" and CREDIT_BLOCK_SOURCE == "oas_buckets":
    print("WARNING: SAMPLE_MODE='long' with CREDIT_BLOCK_SOURCE='oas_buckets' -- the FRED OAS "
          "panel's rolling ~3y truncation binds the joint window, so the long mode is inert "
          "(the v18 lesson). Set CREDIT_BLOCK_SOURCE='moody_2pt' to unlock the pre-2023 history.")
print(f"Aligned joint window      : {stock_prices.index.min().date()} -> {effective_date.date()}  "
      f"({len(stock_prices)} business days)")
print()
print(f"Configured AS_OF_DATE      : {AS_OF_DATE.date()}")
print(f"Effective evaluation date : {effective_date.date()}")
print(f"Equity sample shape       : {stock_prices.shape}")
print(f"Market proxy sample shape : {market_proxy_prices.shape}")
print(f"Rates sample shape        : {key_rates.shape}")
print(f"FF factor sample shape    : {ff_factor_panel.shape}")
print(f"Credit driver panel shape : {credit_spreads.shape}  (columns: {list(credit_spreads.columns)})")
print(f"Credit driver date range  : {credit_spreads.index.min().date()} -> {credit_spreads.index.max().date()}")
print(f"FX sample shape           : {fx_rates.shape}  (pairs: {list(fx_rates.columns)})")
print(f"FX spot at effective date : " + ", ".join(f"{p}={fx_rates[p].iloc[-1]:.4f}" for p in fx_rates.columns))
print(f"Active ERP engine         : {ACTIVE_EQUITY_ENGINE}")

print()
print("Equity universe and sectors:")
display(sector_table)

print("Latest credit-spread curve at the effective date (bps):")
_curve_now = credit_bucket_panel.loc[:effective_date].tail(1)
display((_curve_now.T * 1e4).rename(columns={_curve_now.index[-1]: "OAS_bps"}))

print("Latest cross-asset state at the effective date:")
display(pd.concat([stock_prices.tail(1).T, market_proxy_prices.tail(1).T, key_rates.tail(1).T], axis=0))


In [ ]:
# Visual inspection of raw state variables
plot_time_series(stock_prices, "Adjusted-close prices — broad US equity prototype universe", "Price")
plot_time_series(market_proxy_prices, f"Adjusted-close price — market proxy ({MARKET_PROXY})", "Price")
plot_time_series(key_rates * 100.0, "Treasury key rates — full key-rate grid (%)", "Yield (%)")
plot_time_series(ff_factor_panel[[c for c in ["MKT_RF", "SMB", "HML", "MOM"] if c in ff_factor_panel.columns]],
                 "Daily observable equity factor panel (log-return approximations)",
                 "Daily factor return")


## 2. Risk-driver design and modular ERP engines

The equity block is now **plug-and-play**. The notebook builds several alternative engines and lets the user choose the active one:

- **latent_statistical**: latent common factor extracted from equity excess returns,
- **proxy_market**: observed market ERP proxy via the broad market index,
- **apt_ff3**: observable factor engine based on MKT-RF, SMB, HML,
- **apt_carhart**: observable factor engine based on MKT-RF, SMB, HML, MOM.

Each engine is paired with a compressed idiosyncratic residual block.

Alongside equities, two curve blocks share the same construction: the **Treasury key-rate curve** (daily yield changes → PCA into level/slope/curvature factors) and, new in v7, the **credit-spread curve**. The credit block takes the ICE BofA option-adjusted spreads (OAS) across rating buckets (AAA→CCC), differences them, and compresses them by PCA into a few **credit factors** (overall credit-risk level, IG-vs-HY quality slope, curvature). All blocks — equity factors, equity residual PCs, rates PCs, credit PCs — are stacked into a single joint state, so the contemporaneous **equity-credit dependence** (spreads widening as equity falls) is captured by the joint innovation bootstrap rather than by an imposed correlation.


In [ ]:
# Build transformed risk drivers and alternative equity-premium engines
equity_log_returns = compute_equity_log_returns(stock_prices)
yield_changes = compute_yield_changes(key_rates)
market_log_returns = compute_equity_log_returns(market_proxy_prices)
spread_changes = compute_spread_changes(credit_spreads)
fx_log_returns = compute_fx_log_returns(fx_rates)

common_dates = (
    equity_log_returns.index
    .intersection(yield_changes.index)
    .intersection(market_log_returns.index)
    .intersection(ff_factor_panel.index)
    .intersection(spread_changes.index)
    .intersection(fx_log_returns.index)
)
equity_log_returns = equity_log_returns.loc[common_dates]
yield_changes = yield_changes.loc[common_dates]
spread_changes = spread_changes.loc[common_dates]
fx_log_returns = fx_log_returns.loc[common_dates]
key_rates = key_rates.loc[common_dates]
credit_spreads = credit_spreads.loc[common_dates]
fx_rates = fx_rates.loc[common_dates]
market_proxy_prices = market_proxy_prices.loc[common_dates]
ff_factor_panel = ff_factor_panel.loc[common_dates]

# Issue #6: single risk-free convention. The daily Fama-French RF (already downloaded) is the same
# risk-free that MKT_RF embeds, so we use it to define equity excess returns instead of DGS2/252.
rf_daily = ff_factor_panel["RF"]

latent_equity_block = build_latent_statistical_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

proxy_equity_block = build_proxy_market_equity_block(
    log_returns=equity_log_returns,
    market_proxy_prices=market_proxy_prices,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_ff3_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML"],
    block_name="apt_ff3",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_carhart_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML", "MOM"],
    block_name="apt_carhart",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

equity_engines = {
    "latent_statistical": latent_equity_block,
    "proxy_market": proxy_equity_block,
    "apt_ff3": apt_ff3_equity_block,
    "apt_carhart": apt_carhart_equity_block,
}

if ACTIVE_EQUITY_ENGINE not in equity_engines:
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")

equity_block = equity_engines[ACTIVE_EQUITY_ENGINE]

rates_block = build_rates_factor_model(
    yield_changes=yield_changes,
    explained_var_target=TARGET_RATE_EXPLAINED_VAR,
)

# Credit block: PCA of the rating-bucket OAS curve. Its factors join the joint state, so the
# equity-credit (Merton-type) co-movement is carried by the joint innovation bootstrap.
credit_block = build_credit_factor_model(
    spread_changes=spread_changes,
    explained_var_target=TARGET_CREDIT_EXPLAINED_VAR,
    max_factors=CREDIT_MAX_FACTORS,
)

# Sensitivity bridge (moody_2pt only): per-bucket OLS of daily bucket-OAS changes on the 2-point
# spread changes over the overlap window. It converts SIMULATED 2-point moves into bucket space for
# bond repricing -- sensitivities plus today's real anchor, no fabricated bucket history. The
# walk-forward backtest never touches it (its metric is equity VaR on the native 2-point engine).
credit_bridge = None
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    credit_bridge = estimate_credit_bucket_bridge(
        bucket_panel=credit_bucket_panel,
        spread_changes_2pt=spread_changes,
        anchor_date=effective_date,
    )

# FX block: PCA of the currency-pair daily log-returns. Its factors join the joint state (after the
# credit PCs), so the equity-FX and rates-FX co-movements are carried by the same joint innovation
# bootstrap. With a single pair this is one factor; the block compresses if more pairs are added.
fx_block = build_fx_factor_block(
    fx_log_returns=fx_log_returns,
    explained_var_target=TARGET_FX_EXPLAINED_VAR,
    max_factors=FX_MAX_FACTORS,
)

# Crisis-tail calibration from the long Moody's Baa-Aaa history (decades incl. every postwar crisis).
# This characterizes how often / how severely credit blows out vs the short calm window; the regime
# overlay in the simulator uses it. Computed regardless of CRISIS_MODE (informative even when off).
crisis_calibration = compute_credit_crisis_calibration(
    moodys=moodys_raw,
    horizon_bdays=HORIZON_BDAYS,
    calm_sigma_threshold=CRISIS_SIGMA_THRESHOLD,
)
# Effective regime parameters: config overrides win, else the Moody's-calibrated values.
crisis_prob_eff = CRISIS_PROB if CRISIS_PROB is not None else crisis_calibration["crisis_prob"]
crisis_scale_eff = CRISIS_SCALE if CRISIS_SCALE is not None else crisis_calibration["crisis_scale"]
crisis_pool_eff = crisis_calibration["stress_magnitude_pool"]
print(
    f"Crisis backbone (Moody's Baa-Aaa, {crisis_calibration['history_start'].date()} -> "
    f"{crisis_calibration['history_end'].date()}, {crisis_calibration['n_total_days']} days): "
    f"{crisis_calibration['n_stress_days']} stress days -> crisis_prob={crisis_prob_eff:.4f}/day, "
    f"crisis_scale={crisis_scale_eff:.2f}x, tail dof={crisis_calibration['dof']:.1f}. "
    f"Active CRISIS_MODE={CRISIS_MODE!r}."
)

curve_lsc_levels = build_curve_lsc_factors(key_rates)
curve_lsc_changes = curve_lsc_levels.diff().dropna()

credit_lsc_levels = build_credit_lsc_factors(credit_spreads)
credit_lsc_changes = credit_lsc_levels.diff().dropna()

engine_summary = pd.concat([
    summarize_equity_engine(block) for block in equity_engines.values()
], axis=1).T.sort_values(["mean_asset_R2", "median_asset_R2"], ascending=False)

print(f"Active equity factor block: {equity_block['engine_name']}")
print(f"Rates factors retained   : {rates_block['n_factors']} (from {key_rates.shape[1]} tenors)")
print(f"Credit factors retained  : {credit_block['n_factors']} (from {credit_spreads.shape[1]} "
      + ("Moody's 2-point spread series)" if CREDIT_BLOCK_SOURCE == "moody_2pt" else "rating buckets)"))
print(f"FX factors retained      : {fx_block['n_factors']} (from {fx_rates.shape[1]} currency pair(s))")
print(f"Daily risk-free source   : Fama-French RF (mean {rf_daily.mean():.6f}/day)")
print("Equity engine comparison:")
display(engine_summary)

print("Credit-spread PCA explained variance:")
display(credit_block["explained_ratio"].to_frame("credit_explained_variance_ratio"))

# Economic interpretation of the credit PCs via level / quality-slope / curvature changes
credit_interpretation = pd.concat([credit_block["scores"], credit_lsc_changes], axis=1).dropna().corr()
credit_interpretation = credit_interpretation.loc[credit_block["scores"].columns, credit_lsc_changes.columns]
print("Interpretation of credit PCA scores (corr with level / quality-slope / curvature changes):")
display(credit_interpretation)

if credit_bridge is not None:
    print(f"\nBucket sensitivity bridge (interceptless OLS on {credit_bridge['n_overlap']} overlap days; "
          "gamma maps daily 2-point spread changes to bucket-OAS changes; used for repricing only):")
    _btab = credit_bridge["gamma"].copy()
    _btab["r2"] = credit_bridge["r2"]
    _btab["anchor_bps"] = credit_bridge["current_buckets"] * 1e4
    display(_btab.round({"Aaa_spr": 2, "Baa_spr": 2, "r2": 2, "anchor_bps": 0}))
    print("Read: a +1bp move of the simulated Baa spread widens each bucket by ~gamma_Baa bps around "
          "its real anchor. Declared limitation: a linear calm-window map understates crisis "
          "decompression on the HY end (CCC convexity) -- a repricing-granularity caveat; the credit "
          "DYNAMICS themselves are the real long 2-point history.")


In [ ]:

# Diagnostics on engine structure and factor interpretation
active_beta_table = equity_block["betas"].copy()
active_beta_table.index.name = "asset"

rate_interpretation = pd.concat([rates_block["scores"], curve_lsc_changes], axis=1).dropna().corr()
rate_interpretation = rate_interpretation.loc[rates_block["scores"].columns, curve_lsc_changes.columns]

compare_factor_table = pd.concat([
    proxy_equity_block["factors"].rename(columns={proxy_equity_block["factors"].columns[0]: "market_erp_proxy"}),
    latent_equity_block["factors"].rename(columns={latent_equity_block["factors"].columns[0]: "latent_equity_factor"}),
    ff_factor_panel[[c for c in ["MKT_RF", "MOM"] if c in ff_factor_panel.columns]],
], axis=1).dropna().corr()

print("Active engine beta matrix:")
display(active_beta_table)

print("Active engine asset-level R^2:")
display(equity_block["asset_r2"].to_frame("asset_R2"))

print("Residual equity PCA explained variance (active engine):")
display(equity_block["resid_explained_ratio"].to_frame("resid_explained_variance_ratio"))
display(equity_block["resid_cum_explained_ratio"].to_frame("resid_cum_explained_ratio"))

print("Comparison across observable and latent equity factors:")
display(compare_factor_table)

print("Economic interpretation of rates PCA scores via level / slope / curvature changes:")
display(rate_interpretation)

plot_time_series(
    pd.concat([equity_block["factors"], rates_block["scores"]], axis=1),
    f"Active ERP engine factors and rates PCA scores — {ACTIVE_EQUITY_ENGINE}",
    "Factor value",
)



## 3. Joint state representation and hybrid scenario engine

The active joint state vector stacks:

- the chosen **ERP engine factors**,
- compressed **equity residual factors**,
- compressed **risk-free curve factors**.

Dynamics are modelled through a **VAR(1)**, while innovations are generated with **stationary bootstrap** to avoid relying purely on Gaussian i.i.d. shocks.


In [ ]:
# Joint state vector and dynamic model
drivers = build_joint_driver_matrix(equity_block, rates_block, credit_block, fx_block)
print(f"Joint driver matrix shape: {drivers.shape}")
print("Driver columns:")
print(drivers.columns.tolist())
display(drivers.head())

var_res = build_var1_model(drivers)
print("VAR(1) fitted successfully.")
print(f"Selected lag order: {var_res.k_ar}")
print(f"Number of state variables: {var_res.neqs}")
display(var_res.params)

try:
    print(var_res.summary())
except Exception as err:
    print("VAR summary could not be rendered cleanly.")
    print(f"Reason: {type(err).__name__}: {err}")

# Issue #9: stability check (statsmodels convention: stable iff ALL companion roots have modulus > 1)
print("Companion roots (modulus should be > 1 for stability):")
print(np.abs(var_res.roots))
print(f"VAR(1) is_stable(): {var_res.is_stable()}")
print(f"Min |root|: {np.min(np.abs(var_res.roots)):.4f}")

# Issue #9: is the VAR(1) doing real work, or is risk coming from the bootstrapped innovations?
# Per-equation one-step in-sample R^2 vs a constant-only baseline.
one_step_r2 = var_one_step_r2(var_res, drivers)
print()
print("VAR(1) one-step-ahead in-sample R^2 per driver:")
display(one_step_r2.to_frame("one_step_R2"))
print(
    f"Mean one-step R^2 across drivers: {one_step_r2.mean():.4f}. "
    "If these are near zero, the VAR mean dynamics add little predictive content and the simulated "
    "risk is effectively driven by the bootstrapped joint innovations + factor loadings; the model "
    "could then be simplified (block-bootstrap the demeaned innovations) or regularized."
)

# Option A: precompute the block-bootstrap innovation source (demeaned joint drivers + their mean),
# used by the simulator when DRIVER_DYNAMICS == "block_bootstrap". This drops the VAR(1) mean
# dynamics and resamples the demeaned drivers (equity factors + resid PCs + rate PCs + credit PCs +
# FX PCs) in stationary blocks around their mean.
driver_mean = drivers.mean()
demeaned_drivers = drivers - driver_mean

_k = drivers.shape[1]
_var_params = _k + _k * _k          # k intercepts + k*k AR coefficients
print()
print("Joint-state dynamics (Option A):")
_dyn_compare = pd.Series({
    "active_DRIVER_DYNAMICS": DRIVER_DYNAMICS,
    "state_dimension_k": _k,
    "var1_dynamics_parameters": _var_params,
    "block_bootstrap_dynamics_parameters": 0,
    "mean_one_step_R2": round(float(one_step_r2.mean()), 4),
})
display(_dyn_compare.to_frame("value"))
print(
    f"Active mode: {DRIVER_DYNAMICS}. 'block_bootstrap' resamples the demeaned joint innovations in "
    f"stationary blocks and removes the {_var_params} VAR(1) dynamics parameters; with a mean one-step "
    f"R^2 of {one_step_r2.mean():.4f} the VAR's mean dynamics contribute little, so the two modes give "
    "near-identical one-week distributions. Set DRIVER_DYNAMICS='var1' in the config to recover the "
    "VAR(1) engine and compare the tail metrics."
)

# t-copula (v11): fit a Student-t copula to the joint innovations. Fitted regardless of the active
# mode so the tail-dependence diagnostic is always available; only used by the simulator when
# DRIVER_DYNAMICS == "t_copula". The dof nu controls tail dependence (low nu -> joint extremes
# co-occur; nu -> inf is the Gaussian / linear-correlation case the bootstrap implicitly assumes).
t_copula_model = fit_t_copula(drivers)
print()
print("Joint-innovation tail dependence (Student-t copula, v11):")
_cop = pd.Series({
    "fitted_dof_nu": round(t_copula_model["nu"], 2),
    "avg_abs_pairwise_corr": round(t_copula_model["avg_abs_corr"], 4),
    "avg_pairwise_tail_dependence": round(t_copula_model["avg_tail_dependence"], 4),
    "copula_loglik": round(t_copula_model["loglik"], 1),
    "gaussian_copula_loglik": round(t_copula_model["gaussian_loglik"], 1),
})
display(_cop.to_frame("value"))
print(
    f"Fitted dof nu={t_copula_model['nu']:.1f} (lower = heavier joint tails). The t-copula gains "
    f"{t_copula_model['loglik'] - t_copula_model['gaussian_loglik']:+.0f} log-likelihood over a "
    f"Gaussian copula (nu->inf), and implies an average pairwise tail-dependence coefficient of "
    f"{t_copula_model['avg_tail_dependence']:.3f} vs 0 for the Gaussian case -- i.e. drivers are "
    "jointly extreme more often than the block bootstrap's resampling of observed vectors implies. "
    "Set DRIVER_DYNAMICS='t_copula' in the config to simulate from it (empirical margins, t dependence)."
)

# Asset-level copula (v12): fit the t-copula on the OBSERVABLES (equity log-returns + key-rate changes
# + spread changes + FX log-return) instead of the orthogonal PCA factors. The factor copula above
# had almost no correlation to work with (the PCs are orthogonal by construction), so its tail
# dependence was negligible; the observables carry the real cross-sectional correlation (the curve
# moves together, the credit buckets move together, equities co-move), so this is where tail
# dependence can actually bite. Used by the simulator only when DRIVER_DYNAMICS == "asset_copula".
asset_copula_model = fit_asset_copula(equity_log_returns, yield_changes, spread_changes, fx_log_returns)
print()
print("Factor-level vs asset-level copula (v12):")
_cop_compare = pd.DataFrame({
    "factor_copula": {
        "dimension": t_copula_model["dim"], "fitted_dof_nu": round(t_copula_model["nu"], 1),
        "avg_abs_corr": round(t_copula_model["avg_abs_corr"], 3),
        "avg_tail_dependence": round(t_copula_model["avg_tail_dependence"], 3),
    },
    "asset_copula": {
        "dimension": asset_copula_model["dim"], "fitted_dof_nu": round(asset_copula_model["nu"], 1),
        "avg_abs_corr": round(asset_copula_model["avg_abs_corr"], 3),
        "avg_tail_dependence": round(asset_copula_model["avg_tail_dependence"], 3),
    },
}).T
display(_cop_compare)
print(
    f"The asset-level copula sees average |correlation| {asset_copula_model['avg_abs_corr']:.3f} "
    f"(vs {t_copula_model['avg_abs_corr']:.3f} for the orthogonal factors) and an implied average "
    f"tail dependence of {asset_copula_model['avg_tail_dependence']:.3f} (vs "
    f"{t_copula_model['avg_tail_dependence']:.3f}). This is the reduced-form generator (no factor "
    "structure); set DRIVER_DYNAMICS='asset_copula' to simulate from it and compare the equity "
    "co-exceedance in the validation cell. The validation cell's tail-dependence panel pits the "
    "historical, block-bootstrap, factor-copula and asset-copula joints against each other."
)

# Conditional-volatility (FHS) filter: standardize each factor by its EWMA/GARCH conditional vol so
# the simulator can re-scale by today's vol. Fitted whenever CONDITIONAL_VOL == "ewma_fhs"; the
# current-vol ratio reads off whether the market is calm or stressed RIGHT NOW relative to its history.
if CONDITIONAL_VOL == "ewma_fhs":
    fhs_filter = compute_ewma_fhs_filter(
        drivers.to_numpy(), driver_mean.to_numpy(),
        ewma_lambda=VOL_EWMA_LAMBDA, reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS,
        params_mode=VOL_PARAMS_MODE,
    )
    if fhs_filter.get("per_factor") is not None:
        _pf = pd.DataFrame(fhs_filter["per_factor"], index=drivers.columns)
        _pf["uncond_var_share"] = fhs_filter["uncond_var"] / fhs_filter["uncond_var"].sum()
        # Clustering flag reflects the model ACTUALLY used: a factor conditions on its own vol only
        # if its fitted persistence > 0. A factor gated to constant variance (news weight below the
        # impact gate) keeps persistence 0 and runs flat, even if its in-sample LR is large -- so the
        # flag must key off persistence, not off lr_vs_const, or the count over-states clustering.
        _pf["clustering"] = _pf["persistence"] > 0.0
        _pf = _pf.sort_values("uncond_var_share", ascending=False)
        print()
        print("Per-factor GARCH(1,1) estimates (QML with variance targeting; LR = likelihood-ratio vs "
              "constant variance, indicative threshold ~6; no-evidence factors fall back to constant "
              "variance, i.e. the filter conditions only where the data supports it):")
        _pf_round = {c: d for c, d in {"alpha": 3, "gamma": 3, "persistence": 4,
                                       "half_life_days": 0, "lr_vs_const": 1,
                                       "lr_asym": 1, "uncond_var_share": 3}.items()
                     if c in _pf.columns}
        display(_pf.head(12).round(_pf_round))
        if "gamma" in _pf.columns:
            _nl = int((_pf["gamma"] > 0).sum())
            print(f"{_nl} of {len(_pf)} factors show a leverage effect (gamma > 0, LR_asym >= 3.84): "
                  "for these, negative shocks raise next-day variance by alpha + gamma, positive "
                  "shocks by alpha alone. Factors without asymmetry evidence keep the symmetric fit.")
        _nc = int(_pf["clustering"].sum())
        _ngate = int(((_pf["persistence"] <= 0.0) & (_pf["lr_vs_const"] >= 6.0)).sum())
        print(f"{_nc} of {len(_pf)} factors show volatility clustering (fitted persistence > 0, so the "
              "filter conditions on them); the rest run at constant variance. This is the per-factor "
              "heterogeneity that one fixed half-life (and, before it, one block length) could not "
              "represent -- each clustered factor now decays at its own estimated speed."
              + (f" {_ngate} factor(s) carry a large in-sample LR yet were gated to constant variance: "
                 "their estimated news weight alpha + gamma/2 sits below the impact gate -- "
                 "statistically detectable but too small to be worth modelling. The LR column keeps "
                 "that evidence visible while the clustering flag reports the model actually run."
                 if _ngate else ""))
    _vr = pd.Series(fhs_filter["current_vol_ratio"], index=drivers.columns)
    _eq_factor_cols = [c for c in drivers.columns if str(c).startswith(("MKT", "equity", "Mkt", "eq_"))] or list(drivers.columns[:1])
    print()
    print("Conditional volatility (FHS) active. Current conditional vol vs unconditional, per factor "
          f"(>1 = market currently MORE volatile than its history; news weight {1-VOL_EWMA_LAMBDA:.2f}, "
          f"vol half-life {VOL_REVERSION_HALFLIFE_DAYS:.0f} business days):")
    display(_vr.to_frame("current_vol_ratio").round(3))
    print(
        f"The equity market factor is currently at {_vr.iloc[0]:.2f}x its unconditional volatility, so "
        f"the one-week risk will be scaled {'UP' if _vr.iloc[0] > 1 else 'DOWN'} relative to the "
        "unconditional bootstrap. Over the one-year horizon this conditioning mean-reverts away (the "
        "variance reverts to its unconditional level over the half-life), so the 1Y risk is close to "
        "unconditional. Asset correlations are not fitted: they rise in stress automatically because "
        "the shared factor's vol scales every loading on it together."
    )
else:
    fhs_filter = None

# Politis-White automatic block-length selection. The diagnostic makes the block/FHS interaction
# explicit: on the LEVELS the optimal block is short (returns are nearly serially uncorrelated), but
# on the ABSOLUTE series it is long for the raw innovations (that is the volatility clustering the
# block must otherwise carry) and short again for the FHS standardized residuals (the GARCH filter has
# already absorbed the clustering). So the right data-driven block is short *when FHS is on* and long
# when it is off -- two routes to capturing the same dependence.
def _pw_median(mat):
    return float(np.median(politis_white_block_length(np.asarray(mat))))

_dd = demeaned_drivers.to_numpy()
_pw = {"raw innovations (levels)": _pw_median(_dd),
       "raw innovations |.| (vol clustering)": _pw_median(np.abs(_dd))}
if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
    _z = fhs_filter["standardized_residuals"]
    _pw["FHS standardized residuals (levels)"] = _pw_median(_z)
    _pw["FHS standardized |.| (vol clustering)"] = _pw_median(np.abs(_z))

if BOOTSTRAP_BLOCK_MODE == "politis_white":
    if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
        effective_block_length = max(1, int(round(_pw["FHS standardized residuals (levels)"])))
        _blk_src = "Politis-White on FHS standardized residuals"
    else:
        effective_block_length = max(1, int(round(_pw["raw innovations |.| (vol clustering)"])))
        _blk_src = "Politis-White on |raw innovations| (captures vol clustering; no FHS active)"
else:
    effective_block_length = BOOTSTRAP_EXPECTED_BLOCK
    _blk_src = f"fixed BOOTSTRAP_EXPECTED_BLOCK={BOOTSTRAP_EXPECTED_BLOCK}"

print()
print("Politis-White optimal expected block length (median across factors):")
display(pd.Series(_pw, name="optimal_block_bdays").round(2).to_frame())
print(
    f"Block length used by the bootstrap: {effective_block_length} business days ({_blk_src}). "
    "On the levels the optimum is short because returns are nearly serially uncorrelated; the long "
    "block on the |raw innovations| is the volatility clustering, which the FHS filter absorbs -- "
    "hence the FHS standardized residuals admit a short block. Set BOOTSTRAP_BLOCK_MODE='politis_white' "
    "to use the data-driven value; 'fixed' keeps the hard-coded block."
)

# ---- Level mean reversion (OU / Vasicek) on rate and spread levels --------
# Estimated on the same sample the bootstrap resamples from. Where a series is
# gated in, the simulator replaces the bootstrapped random-walk drift with an
# explicit daily pull kappa * (theta - level); non-gated series keep the
# random walk exactly, so reversion is applied only where the data supports it.
ou_rates = None
ou_credit = None
if LEVEL_MEAN_REVERSION == "ou":
    ou_rates = estimate_ou_level_params(key_rates.to_numpy(), tstat_gate=OU_TSTAT_GATE)
    if credit_spreads is not None and credit_spreads.shape[1] > 0:
        ou_credit = estimate_ou_level_params(credit_spreads.to_numpy(), tstat_gate=OU_TSTAT_GATE)
    print()
    print("Ornstein-Uhlenbeck level mean reversion (gate: OLS slope < 0 AND t-stat <= "
          f"{OU_TSTAT_GATE:.2f}, the Dickey-Fuller 5% critical value -- under the unit-root "
          "null the OLS t-stat is not t-distributed, so an ordinary -2 gate would find "
          "spurious reversion on short samples). Non-gated series keep the random walk:")
    _ou_tabs = [("rates", ou_rates, key_rates.columns)]
    if ou_credit is not None:
        _ou_tabs.append(("credit spreads", ou_credit, credit_spreads.columns))
    for _nm, _ou, _cols in _ou_tabs:
        _t = pd.DataFrame(_ou, index=_cols)[["kappa", "theta", "half_life_days", "tstat", "gated"]]
        _ng = int(_t["gated"].sum())
        print(f"\n{_nm}: {_ng} of {len(_t)} series gated into mean reversion "
              "(theta in the same decimal units as the levels, e.g. 0.043 = 4.3%; half-life in "
              "business days):")
        display(_t.round({"kappa": 4, "theta": 3, "half_life_days": 0, "tstat": 2, "gated": 0}))


In [ ]:
# Simulate one-week joint scenarios under the active ERP engine
current_prices = stock_prices.loc[effective_date]
current_key_rates = key_rates.loc[effective_date]
current_spreads = credit_spreads.loc[effective_date]
# The anchor in BUCKET space for downstream cells: with the bridge active the simulator returns
# bucket scenarios around the real bucket anchor, so "scenario minus current" must use that anchor.
bucket_anchor_spreads = credit_bridge["current_buckets"] if credit_bridge is not None else current_spreads
current_fx = fx_rates.loc[effective_date]

# Issue #6: constant daily risk-free for the excess->total reconstruction, taken as the latest
# Fama-French RF (same convention used to define the excess returns). Issue #11: re-inject the
# idiosyncratic residual variance dropped by the PCA compression as iid noise. Credit block: the
# rating-bucket OAS curve is simulated jointly and returned as credit_spread_scenarios.
current_daily_rf = float(ff_factor_panel["RF"].iloc[-1])

sim = simulate_joint_horizon(
    var_res=var_res,
    resid_hist=var_res.resid,
    last_state=drivers.iloc[-1].to_numpy(),
    equity_block=equity_block,
    rates_block=rates_block,
    current_prices=current_prices,
    current_key_rates=current_key_rates,
    horizon_bdays=HORIZON_BDAYS,
    n_scenarios=N_SCENARIOS,
    avg_block_length=effective_block_length,
    rng_seed=RNG_SEED,
    daily_rf=current_daily_rf,
    resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
    credit_block=credit_block,
    current_spreads=current_spreads,
    fx_block=fx_block,
    current_fx=current_fx,
    dynamics=DRIVER_DYNAMICS,
    demeaned_drivers=demeaned_drivers,
    driver_mean=driver_mean,
    crisis_mode=CRISIS_MODE,
    crisis_prob=crisis_prob_eff,
    crisis_scale=crisis_scale_eff,
    crisis_pool=crisis_pool_eff,
    copula=t_copula_model,
    asset_copula=asset_copula_model,
    fhs_filter=fhs_filter,
    ou_rates=ou_rates, ou_credit=ou_credit,
    credit_bridge=credit_bridge,
)

stock_excess_return_scenarios = sim["stock_excess_return_scenarios"]
stock_log_return_scenarios = sim["stock_log_return_scenarios"]
stock_price_scenarios = sim["stock_price_scenarios"]
key_rate_scenarios = sim["key_rate_scenarios"]
credit_spread_scenarios = sim["credit_spread_scenarios"]
fx_spot_scenarios = sim["fx_spot_scenarios"]


In [ ]:

# Marginal forecast views
plot_histogram(
    stock_excess_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Forecast distribution — {REPRESENTATIVE_EQUITY} one-week excess log-return",
    "One-week excess log-return",
)

plot_histogram(
    key_rate_scenarios["DGS10"] * 100.0,
    "Forecast distribution — 10Y Treasury yield at horizon",
    "Yield (%)",
)

display(loss_summary_from_pnl(stock_excess_return_scenarios[REPRESENTATIVE_EQUITY]).to_frame(f"{REPRESENTATIVE_EQUITY}_1w_excess_return"))
display(distribution_summary(key_rate_scenarios["DGS10"]).to_frame("DGS10_horizon_yield"))


In [ ]:
# Distributional and dependence validation (issues #1a, #1c; plus equity-credit dependence)
# A joint scenario engine must be judged on its dependence and tail calibration, not only on a
# single-name standard deviation. We compare simulated one-week behaviour with historical 5-day
# behaviour along seven dimensions (risk scaling, kurtosis, cross-equity correlation, equity-rates
# correlation, equity-credit correlation, credit-spread scaling, and FX risk-scaling / equity-FX
# correlation); the VaR coverage / PIT calibration is handled separately by the walk-forward
# backtest below.

hist_5d_stock = rolling_horizon_log_returns(equity_log_returns, HORIZON_BDAYS)
hist_5d_yield = rolling_horizon_yield_changes(yield_changes, HORIZON_BDAYS)
sim_d10 = key_rate_scenarios["DGS10"] - current_key_rates["DGS10"]

# (1) Risk scaling: simulated one-week std vs historical rolling 5-day std
scaling_table = pd.DataFrame({
    "hist_5d_std": [hist_5d_stock[REPRESENTATIVE_EQUITY].std(), hist_5d_yield["DGS10"].std()],
    "sim_1w_std": [stock_log_return_scenarios[REPRESENTATIVE_EQUITY].std(), sim_d10.std()],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
scaling_table["sim/hist"] = scaling_table["sim_1w_std"] / scaling_table["hist_5d_std"]
print("Risk scaling (simulated one-week vs historical 5-day std):")
display(scaling_table)

# (2) Tail behaviour: excess kurtosis (pandas .kurt() is the Fisher / excess kurtosis)
def _exkurt(x):
    return float(pd.Series(x).kurt())

kurt_table = pd.DataFrame({
    "hist_5d_excess_kurtosis": [_exkurt(hist_5d_stock[REPRESENTATIVE_EQUITY]), _exkurt(hist_5d_yield["DGS10"])],
    "sim_1w_excess_kurtosis": [_exkurt(stock_log_return_scenarios[REPRESENTATIVE_EQUITY]), _exkurt(sim_d10)],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
print("Tail behaviour (excess kurtosis, simulated vs historical 5-day):")
display(kurt_table)

# (3) Cross-equity dependence: simulated vs historical 5-day correlation matrices
sim_corr = stock_log_return_scenarios.corr()
hist_corr = hist_5d_stock.corr()

def _avg_offdiag(corr_df):
    arr = corr_df.to_numpy().copy()
    np.fill_diagonal(arr, np.nan)
    return float(np.nanmean(arr))

cross_equity_compare = pd.Series({
    "sim_avg_cross_equity_corr": _avg_offdiag(sim_corr),
    "hist_avg_cross_equity_corr": _avg_offdiag(hist_corr),
    "max_abs_pairwise_corr_diff": float(np.nanmax(np.abs(sim_corr.to_numpy() - hist_corr.to_numpy()))),
})
print("Cross-equity dependence (simulated vs historical 5-day):")
display(cross_equity_compare.to_frame("value"))

# (4) Equity-vs-rates dependence: corr(equity 5-day return, 5-day change in DGS10)
sim_eq_rate_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_d10)[0, 1]) for t in TICKERS}
)
hist_joint = pd.concat([hist_5d_stock, hist_5d_yield["DGS10"].rename("DGS10_chg")], axis=1).dropna()
hist_eq_rate_corr = pd.Series({t: float(hist_joint[t].corr(hist_joint["DGS10_chg"])) for t in TICKERS})
eq_rate_compare = pd.DataFrame({"sim": sim_eq_rate_corr, "hist": hist_eq_rate_corr})
eq_rate_compare.loc["AVERAGE"] = eq_rate_compare.mean()
print("Equity vs DGS10-change correlation (simulated vs historical 5-day):")
display(eq_rate_compare)

# (5) Equity-credit dependence (the Merton-type link): corr(equity 5-day return, 5-day change in
# the representative-rating OAS). The joint innovation bootstrap should reproduce the empirical
# (negative) co-movement: when equity falls, credit spreads widen.
hist_5d_spread = rolling_horizon_yield_changes(spread_changes, HORIZON_BDAYS)
# Validate the engine against ITS OWN history, i.e. in the space it actually simulates: with the
# bucket bridge active the native credit panel is the Moody's 2-point curve (preserved by the
# simulator under "credit_spread_scenarios_raw"); the bridged bucket scenarios are a repricing view.
_val_credit_scen = sim.get("credit_spread_scenarios_raw", credit_spread_scenarios)
_val_rep = REPRESENTATIVE_RATING if REPRESENTATIVE_RATING in _val_credit_scen.columns else _val_credit_scen.columns[-1]
sim_dspread_rep = _val_credit_scen[_val_rep] - current_spreads[_val_rep]

sim_eq_credit_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_dspread_rep)[0, 1]) for t in TICKERS}
)
hist_ec = pd.concat(
    [hist_5d_stock, hist_5d_spread[_val_rep].rename("spread_chg")], axis=1
).dropna()
hist_eq_credit_corr = pd.Series({t: float(hist_ec[t].corr(hist_ec["spread_chg"])) for t in TICKERS})
eq_credit_compare = pd.DataFrame({"sim": sim_eq_credit_corr, "hist": hist_eq_credit_corr})
eq_credit_compare.loc["AVERAGE"] = eq_credit_compare.mean()
print(f"Equity vs {_val_rep} credit-change correlation (simulated vs historical 5-day):")
display(eq_credit_compare)

# (6) Credit-spread risk scaling: simulated one-week vs historical 5-day std of the OAS change
credit_scaling = pd.DataFrame({
    "hist_5d_std": [hist_5d_spread[r].std() for r in _val_credit_scen.columns],
    "sim_1w_std": [(_val_credit_scen[r] - current_spreads[r]).std() for r in _val_credit_scen.columns],
}, index=list(_val_credit_scen.columns))
credit_scaling["sim/hist"] = credit_scaling["sim_1w_std"] / credit_scaling["hist_5d_std"]
print("Credit-spread risk scaling (simulated one-week vs historical 5-day std, by rating):")
display(credit_scaling)

# (7) FX block: simulated one-week vs historical 5-day std of the FX log-return, plus the equity-FX
# dependence (corr of the equity 5-day return with the 5-day FX log-return). Same sim-vs-hist logic
# as the other blocks; for a single pair this checks that the FX factor is scaled correctly and that
# its (weak) co-movement with equities is reproduced.
hist_5d_fx = rolling_horizon_log_returns(fx_log_returns, HORIZON_BDAYS)
sim_fx_ret = {p: np.log(fx_spot_scenarios[p] / current_fx[p]) for p in fx_spot_scenarios.columns}

fx_scaling = pd.DataFrame({
    "hist_5d_std": [hist_5d_fx[p].std() for p in fx_spot_scenarios.columns],
    "sim_1w_std": [sim_fx_ret[p].std() for p in fx_spot_scenarios.columns],
}, index=list(fx_spot_scenarios.columns))
fx_scaling["sim/hist"] = fx_scaling["sim_1w_std"] / fx_scaling["hist_5d_std"]
print("FX risk scaling (simulated one-week vs historical 5-day std of the FX log-return):")
display(fx_scaling)

rep_fx = list(fx_spot_scenarios.columns)[0]
sim_eq_fx_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_fx_ret[rep_fx])[0, 1]) for t in TICKERS}
)
hist_ef = pd.concat([hist_5d_stock, hist_5d_fx[rep_fx].rename("fx_ret")], axis=1).dropna()
hist_eq_fx_corr = pd.Series({t: float(hist_ef[t].corr(hist_ef["fx_ret"])) for t in TICKERS})
eq_fx_compare = pd.DataFrame({"sim": sim_eq_fx_corr, "hist": hist_eq_fx_corr})
eq_fx_compare.loc["AVERAGE"] = eq_fx_compare.mean()
print(f"Equity vs {rep_fx} 5-day-return correlation (simulated vs historical 5-day):")
display(eq_fx_compare)

# (8) Crisis-tail overlay effect. The seven checks above validate the engine against the CALM
# 3-year sample. The crisis overlay is an opt-in stress layer that deliberately fattens the tail
# beyond that calm sample (the sample contains no real crisis). Here we quantify it: re-simulate the
# equal-weight equity one-week loss under the calm engine and under both crisis modes, calibrated
# from the long Moody's Baa-Aaa history, and compare the 95% ES and the excess kurtosis. This is a
# demonstration of the overlay, independent of the active CRISIS_MODE used elsewhere in the notebook.
def _eq_loss_tail(crisis_mode):
    s = simulate_joint_horizon(
        var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
        equity_block=equity_block, rates_block=rates_block,
        current_prices=current_prices, current_key_rates=current_key_rates,
        horizon_bdays=HORIZON_BDAYS, n_scenarios=8000, avg_block_length=effective_block_length,
        rng_seed=RNG_SEED, daily_rf=current_daily_rf,
        resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
        credit_block=credit_block, current_spreads=current_spreads,
        fx_block=fx_block, current_fx=current_fx,
        dynamics=DRIVER_DYNAMICS, demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
        crisis_mode=crisis_mode, crisis_prob=crisis_prob_eff, crisis_scale=crisis_scale_eff,
        crisis_pool=crisis_pool_eff, copula=t_copula_model, asset_copula=asset_copula_model, fhs_filter=fhs_filter,
        ou_rates=ou_rates, ou_credit=ou_credit,
    )
    port = s["stock_log_return_scenarios"].mean(axis=1)   # equal-weight equity one-week return
    loss = -port
    thr = loss.quantile(0.95)
    return {"std": float(port.std()), "ES_95_loss": float(loss[loss >= thr].mean()),
            "excess_kurtosis": float(port.kurtosis()),
            "worst_1pct": float(loss.quantile(0.99))}

crisis_compare = pd.DataFrame({m: _eq_loss_tail(m) for m in ["none", "regime_jump", "moody_backbone"]}).T
crisis_compare["ES_uplift_vs_calm"] = crisis_compare["ES_95_loss"] / crisis_compare.loc["none", "ES_95_loss"]
print(
    f"Crisis-tail overlay effect on the equal-weight equity one-week loss "
    f"(Moody's-calibrated: crisis_prob={crisis_prob_eff:.4f}/day, scale={crisis_scale_eff:.2f}x):"
)
display(crisis_compare)

# (9) Tail dependence: do equities go to their joint extremes together more than linear correlation
# implies? We measure the lower-tail co-exceedance -- the probability two names are BOTH below their
# 5% quantile, averaged over pairs, expressed as P(both in 5% tail) / 0.05 (independence -> 0.05,
# comonotone -> 1) -- for the historical 5-day returns, the block-bootstrap sim, and the t-copula sim
# (both crisis off). The block bootstrap resamples observed joint vectors, so it tracks the empirical
# dependence; the t-copula fits a parametric tail-dependent structure that can extrapolate joint
# extremes beyond those observed in the short sample.
def _lower_tail_coexceedance(stock_returns_df, q=0.05):
    R = stock_returns_df.rank(pct=True).to_numpy()
    below = (R < q).astype(float)
    co = (below.T @ below) / below.shape[0]
    iu = np.triu_indices(below.shape[1], k=1)
    return float(co[iu].mean() / q)

def _sim_equity(dyn):
    s = simulate_joint_horizon(
        var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
        equity_block=equity_block, rates_block=rates_block, current_prices=current_prices,
        current_key_rates=current_key_rates, horizon_bdays=HORIZON_BDAYS, n_scenarios=8000,
        avg_block_length=effective_block_length, rng_seed=RNG_SEED, daily_rf=current_daily_rf,
        resid_idio_std=equity_block.get("resid_idiosyncratic_std"), credit_block=credit_block,
        current_spreads=current_spreads, fx_block=fx_block, current_fx=current_fx,
        dynamics=dyn, demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
        crisis_mode="none", copula=t_copula_model, asset_copula=asset_copula_model, fhs_filter=fhs_filter,
        ou_rates=ou_rates, ou_credit=ou_credit,
    )
    return s["stock_log_return_scenarios"]

tail_dep = pd.Series({
    "historical_5d": _lower_tail_coexceedance(hist_5d_stock[TICKERS]),
    "block_bootstrap": _lower_tail_coexceedance(_sim_equity("block_bootstrap")),
    "factor_t_copula": _lower_tail_coexceedance(_sim_equity("t_copula")),
    "asset_copula": _lower_tail_coexceedance(_sim_equity("asset_copula")),
})
print(
    f"Lower-tail co-exceedance of equities (P(pair both in 5% tail)/0.05; 0.05 = independence) -- "
    f"factor copula nu={t_copula_model['nu']:.1f}, asset copula nu={asset_copula_model['nu']:.1f}:"
)
display(tail_dep.to_frame("coexceedance"))
print(
    "The block bootstrap and the asset-level copula are both expected to track the historical "
    "equity co-tail (the bootstrap resamples observed joint vectors; the asset copula fits the "
    "dependence where the real correlation lives), whereas the factor-level copula -- fit on the "
    "orthogonal PCs -- understates it because the equity co-tail is a shared-loadings effect, not a "
    "factor-innovation-copula effect. The asset copula can also extrapolate joint extremes beyond "
    "the observed sample, which the bootstrap cannot."
)

# Marginal distribution shape, historical vs simulated
plot_histogram(
    hist_5d_stock[REPRESENTATIVE_EQUITY],
    f"Historical rolling 5-day {REPRESENTATIVE_EQUITY} log-returns",
    "5-day log-return",
)
plot_histogram(
    stock_log_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Simulated one-week {REPRESENTATIVE_EQUITY} log-returns",
    "one-week log-return",
)


# Conditional-correlation payoff (FHS only): the headline claim is that asset correlations rise in
# stress *for free*, because the shared SYSTEMATIC factors' conditional vol scales every loading on
# them together -- no asset correlation matrix is fitted. To isolate that mechanism we scale only the
# systematic equity factors' (market/size/value/momentum) current conditional variance, holding the
# idiosyncratic residual PCs and the name-specific noise at their current level: a real crisis is
# precisely the common factor blowing out relative to idiosyncratic risk, which is what lifts the
# systematic share of every name's return and pulls correlations up. (Scaling *all* factors uniformly
# would understate this, since it scales idiosyncratic risk in lockstep and only shrinks the small
# fixed noise term.)
if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
    _sys_eq_idx = [i for i, c in enumerate(drivers.columns)
                   if not str(c).startswith(("resid_pc", "rate_pc", "credit_pc", "fx"))]
    _sys_eq_names = [str(drivers.columns[i]) for i in _sys_eq_idx]

    def _sim_equity_vol(sys_scale):
        f2 = dict(fhs_filter)
        cv = np.array(fhs_filter["cond_var_now"], dtype=float).copy()
        cv[_sys_eq_idx] = cv[_sys_eq_idx] * sys_scale      # scale ONLY the systematic equity factors
        f2["cond_var_now"] = cv
        s = simulate_joint_horizon(
            var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
            equity_block=equity_block, rates_block=rates_block, current_prices=current_prices,
            current_key_rates=current_key_rates, horizon_bdays=HORIZON_BDAYS, n_scenarios=8000,
            avg_block_length=effective_block_length, rng_seed=RNG_SEED, daily_rf=current_daily_rf,
            resid_idio_std=equity_block.get("resid_idiosyncratic_std"), credit_block=credit_block,
            current_spreads=current_spreads, fx_block=fx_block, current_fx=current_fx,
            dynamics="block_bootstrap", demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
            crisis_mode="none", fhs_filter=f2,
            ou_rates=ou_rates, ou_credit=ou_credit,
        )["stock_log_return_scenarios"]
        corr = s.corr().to_numpy()
        return float(corr[np.triu_indices(corr.shape[0], 1)].mean())
    _calm = _sim_equity_vol(0.25)      # systematic factors a quarter of their current variance
    _now = _sim_equity_vol(1.0)        # current systematic state
    _stress = _sim_equity_vol(9.0)     # systematic factors at a 3x-vol stress
    print()
    print(f"Conditional correlation (FHS): average cross-equity correlation as ONLY the systematic "
          f"equity factors {_sys_eq_names} are scaled from calm to stressed (idiosyncratic risk held "
          "fixed), with no correlation matrix anywhere in the model:")
    display(pd.Series({"calm (systematic x0.25 var)": _calm, "current state": _now,
                       "stressed (systematic x9 var)": _stress},
                      name="avg_cross_equity_corr").to_frame().round(3))
    print(
        f"Cross-equity correlation rises from {_calm:.3f} (calm) to {_stress:.3f} (stressed) purely "
        "because the shared systematic factors' higher conditional volatility lifts the systematic "
        "share of each name's return relative to its (fixed) idiosyncratic risk. This is the "
        "'correlations go to one in a crash' behaviour, reproduced structurally rather than by "
        "fitting a conditional correlation matrix on the asset pairs."
    )


### 3b. Walk-forward VaR backtest (coverage and calibration)

A single as-of date cannot validate a scenario engine. Here the engine is **refit on an expanding window** at a panel of past as-of dates; at each date we simulate the one-week distribution of an **equal-weight equity** portfolio, take its 95% VaR, and compare it with the **realized** forward one-week return. We then run the **Kupiec proportion-of-failures** test (unconditional coverage) and the **Christoffersen independence** test (no clustering of breaches), and inspect the **PIT / rank histogram** (which should be roughly uniform if the predictive distribution is well calibrated).

This is the most expensive cell because it refits the full pipeline at each date on live data; the defaults (`BACKTEST_N_DATES`, `BACKTEST_N_SCENARIOS`) are deliberately modest. Increase them for more statistical power.


In [ ]:
# Walk-forward VaR/ES backtest. Kupiec (unconditional coverage), Christoffersen (independence) and
# their JOINT conditional-coverage test, an Acerbi-Szekely expected-shortfall test, a PIT histogram,
# and -- with BACKTEST_MODE="full" -- coverage sliced by volatility regime and by calendar year.
# The engine is refit on an expanding, point-in-time window at each as-of date (no lookahead).

def _build_active_equity_block(log_returns, market_prices, ff_panel, rf_d):
    if ACTIVE_EQUITY_ENGINE == "latent_statistical":
        return build_latent_statistical_equity_block(log_returns, rf_d, TARGET_RESID_EXPLAINED_VAR)
    if ACTIVE_EQUITY_ENGINE == "proxy_market":
        return build_proxy_market_equity_block(log_returns, market_prices, rf_d, TARGET_RESID_EXPLAINED_VAR)
    if ACTIVE_EQUITY_ENGINE == "apt_ff3":
        return build_apt_equity_block(log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML"], "apt_ff3", TARGET_RESID_EXPLAINED_VAR)
    if ACTIVE_EQUITY_ENGINE == "apt_carhart":
        return build_apt_equity_block(log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML", "MOM"], "apt_carhart", TARGET_RESID_EXPLAINED_VAR)
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")


def _fit_engine_for_asof(asof):
    eff = choose_effective_asof(stock_prices_raw, market_proxy_raw, rates_raw, ff_factor_panel_raw, asof, credit=credit_driver_raw)
    # FX is intentionally omitted from the refitted backtest engine: the backtest metric is
    # equal-weight EQUITY VaR coverage, which does not involve the FX forward, so adding the FX
    # factor here would only cost time. align now returns a 6-tuple; the FX slot is unused (None).
    sp, mp, kr, ff, cr, _fx = align_market_data(stock_prices_raw, market_proxy_raw, rates_raw, ff_factor_panel_raw, eff, credit=credit_driver_raw)
    # Same contiguity rule as the main data cell: a refitted window must never take a daily change
    # across an internal data hole either (no-op when there is none, i.e. always in "short" mode).
    _ws, _, _ = contiguous_window_start(sp.index, LONG_GAP_TOLERANCE_DAYS)
    if _ws is not None:
        sp, mp, kr, ff, cr = sp.loc[_ws:], mp.loc[_ws:], kr.loc[_ws:], ff.loc[_ws:], cr.loc[_ws:]
    lr = compute_equity_log_returns(sp)
    dy = compute_yield_changes(kr)
    mlr = compute_equity_log_returns(mp)
    ds = compute_spread_changes(cr)
    cd = lr.index.intersection(dy.index).intersection(mlr.index).intersection(ff.index).intersection(ds.index)
    lr, dy, kr, mp, ff, ds, cr = lr.loc[cd], dy.loc[cd], kr.loc[cd], mp.loc[cd], ff.loc[cd], ds.loc[cd], cr.loc[cd]
    rf_d = ff["RF"]
    eb = _build_active_equity_block(lr, mp, ff, rf_d)
    rb = build_rates_factor_model(dy, TARGET_RATE_EXPLAINED_VAR)
    cb = build_credit_factor_model(ds, TARGET_CREDIT_EXPLAINED_VAR, max_factors=CREDIT_MAX_FACTORS)
    drv = build_joint_driver_matrix(eb, rb, cb)
    vr = build_var1_model(drv)
    return eff, sp, kr, cr, eb, rb, cb, drv, vr, float(rf_d.iloc[-1]), lr, dy, ds


def _trailing_annualized_vol(eff_pos, lookback=21):
    """Equal-weight equity realized vol over the `lookback` days ENDING at the as-of date.

    Known at the as-of date by construction, so labelling regimes with it adds no lookahead to the
    VaR itself (the tercile cut-points below are a diagnostic partition, not an input to a forecast).
    """
    lo = max(0, eff_pos - lookback)
    px = stock_prices_raw[TICKERS].iloc[lo:eff_pos + 1]
    if len(px) < 6:
        return np.nan
    r = np.log(px / px.shift(1)).mean(axis=1).dropna()
    return float(r.std() * np.sqrt(252.0)) if len(r) >= 5 else np.nan


# Choose the panel of as-of dates from the trading calendar, ensuring a forward window exists.
_cal = stock_prices_raw.index
_elig = np.where(np.asarray(_cal <= AS_OF_DATE))[0]
_elig = _elig[_elig + HORIZON_BDAYS < len(_cal)]
_thinned_by = 1
if BACKTEST_MODE == "full":
    # Walk the WHOLE aligned window, starting once BACKTEST_MIN_TRAIN_DAYS of training data exist.
    _aligned = stock_prices.index
    _first_ok = _aligned[min(int(BACKTEST_MIN_TRAIN_DAYS), len(_aligned) - 1)]
    _cand = [p for p in _elig if _cal[p] >= _first_ok][::BACKTEST_STEP_BDAYS]
    if BACKTEST_MAX_DATES is not None and len(_cand) > int(BACKTEST_MAX_DATES):
        _thinned_by = int(np.ceil(len(_cand) / float(BACKTEST_MAX_DATES)))
        _cand = _cand[::_thinned_by]      # even thinning: spacing grows, windows stay non-overlapping
    _sel_positions = sorted(_cand)
else:
    _sel_positions = sorted(list(_elig[::-1][::BACKTEST_STEP_BDAYS][:BACKTEST_N_DATES]))
backtest_asof_dates = [_cal[p] for p in _sel_positions]

print(f"Walk-forward backtest [{BACKTEST_MODE}]: {len(backtest_asof_dates)} as-of dates, "
      f"{BACKTEST_N_SCENARIOS} scenarios each, equal-weight equity portfolio, 95% VaR/ES.")
print(f"From {backtest_asof_dates[0].date()} to {backtest_asof_dates[-1].date()} "
      f"(step {BACKTEST_STEP_BDAYS * _thinned_by} business days). This refits the engine at each date.")
if BACKTEST_MODE == "full":
    _yrs = (backtest_asof_dates[-1] - backtest_asof_dates[0]).days / 365.25
    print(f"Spanning {_yrs:.1f} years of point-in-time refits"
          + (f"; eligible dates thinned {_thinned_by}x to respect BACKTEST_MAX_DATES={BACKTEST_MAX_DATES}"
             if _thinned_by > 1 else "")
          + ". This is the expensive path: expect minutes, not seconds.")
print()

bt_records = []
bt_scenarios = []
for j, asof in enumerate(backtest_asof_dates):
    try:
        eff, sp, kr, cr, eb, rb, cb, drv, vr, rf_last, lr_bt, dy_bt, ds_bt = _fit_engine_for_asof(asof)
        # Option A: honour DRIVER_DYNAMICS in the refitted backtest engine too, so the coverage test
        # reflects the same generative mechanism as the main engine. (FX is still omitted here: the
        # backtest metric is equal-weight EQUITY VaR, which does not involve the FX forward.)
        drv_mean = drv.mean()
        drv_demeaned = drv - drv_mean
        bt_copula = fit_t_copula(drv) if DRIVER_DYNAMICS == "t_copula" else None
        bt_asset_copula = fit_asset_copula(lr_bt, dy_bt, ds_bt) if DRIVER_DYNAMICS == "asset_copula" else None
        bt_fhs = (compute_ewma_fhs_filter(drv.to_numpy(), drv_mean.to_numpy(),
                                          ewma_lambda=VOL_EWMA_LAMBDA, reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS,
                                          params_mode=VOL_PARAMS_MODE)
                  if CONDITIONAL_VOL == "ewma_fhs" else None)
        # Per-asof OU refit on the SAME window the bootstrap resamples from (no lookahead):
        # the Dickey-Fuller gate re-decides at every date, so early windows may honestly
        # reject mean reversion that later, longer windows admit.
        bt_ou_r = (estimate_ou_level_params(kr.to_numpy(), tstat_gate=OU_TSTAT_GATE)
                   if LEVEL_MEAN_REVERSION == "ou" else None)
        bt_ou_c = (estimate_ou_level_params(cr.to_numpy(), tstat_gate=OU_TSTAT_GATE)
                   if (LEVEL_MEAN_REVERSION == "ou" and cr is not None and cr.shape[1] > 0) else None)
        sim_bt = simulate_joint_horizon(
            var_res=vr, resid_hist=vr.resid, last_state=drv.iloc[-1].to_numpy(),
            equity_block=eb, rates_block=rb,
            current_prices=sp.loc[eff], current_key_rates=kr.loc[eff],
            horizon_bdays=HORIZON_BDAYS, n_scenarios=BACKTEST_N_SCENARIOS,
            avg_block_length=effective_block_length, rng_seed=RNG_SEED + j,
            daily_rf=rf_last, resid_idio_std=eb.get("resid_idiosyncratic_std"),
            credit_block=cb, current_spreads=cr.loc[eff],
            dynamics=DRIVER_DYNAMICS, demeaned_drivers=drv_demeaned, driver_mean=drv_mean,
            crisis_mode=CRISIS_MODE, crisis_prob=crisis_prob_eff, crisis_scale=crisis_scale_eff,
            crisis_pool=crisis_pool_eff, copula=bt_copula, asset_copula=bt_asset_copula, fhs_filter=bt_fhs,
            ou_rates=bt_ou_r, ou_credit=bt_ou_c,
        )
        # equal-weight equity portfolio one-week return = mean of single-name cumulative log returns
        port_sim = sim_bt["stock_log_return_scenarios"][TICKERS].mean(axis=1)
        var95 = float(-port_sim.quantile(0.05))
        # The ES test needs the whole predictive distribution, not just its 5% quantile.
        es95 = sample_expected_shortfall(port_sim.to_numpy(), 0.05)

        eff_pos = stock_prices_raw.index.get_loc(eff)
        fwd_pos = eff_pos + HORIZON_BDAYS
        if fwd_pos >= len(stock_prices_raw.index):
            continue
        p0 = stock_prices_raw[TICKERS].iloc[eff_pos]
        p1 = stock_prices_raw[TICKERS].iloc[fwd_pos]
        realized = float(np.log(p1 / p0).mean())

        bt_records.append({
            "asof": eff,
            "VaR_95": var95,
            "ES_95": es95,
            "realized_1w_return": realized,
            "violation": int(realized < -var95),
            "pit": float((port_sim <= realized).mean()),
            "trailing_vol_21d": _trailing_annualized_vol(eff_pos),
        })
        bt_scenarios.append(port_sim.to_numpy(dtype=float))
    except Exception as err:
        print(f"  [skip {pd.Timestamp(asof).date()}] {type(err).__name__}: {err}")
        continue

    if (j + 1) % max(10, len(backtest_asof_dates) // 10) == 0:
        print(f"  ...processed {j + 1}/{len(backtest_asof_dates)} as-of dates")

bt = pd.DataFrame(bt_records)
bt_scenario_matrix = np.vstack(bt_scenarios) if bt_scenarios else None
print(f"\nUsable backtest observations: {len(bt)}")

if len(bt) >= 10:
    _cov = coverage_summary(bt["violation"].to_numpy(), 0.05)

    # ---- Expected-shortfall backtest (Acerbi-Szekely 2014) -------------------------------------
    # The portfolio layer MINIMIZES CVaR, so validating only the VaR left the objective function
    # itself untested. Its own RNG stream, so the engine's scenarios are never perturbed.
    _rng_es = np.random.default_rng(RNG_SEED + 4242)
    _es_args = (bt["realized_1w_return"].to_numpy(), bt["VaR_95"].to_numpy(),
                bt["ES_95"].to_numpy(), bt_scenario_matrix)
    es_test2 = acerbi_szekely_es_test(*_es_args, alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                      rng=_rng_es, which=2)
    es_test1 = acerbi_szekely_es_test(*_es_args, alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                      rng=np.random.default_rng(RNG_SEED + 4243), which=1)

    summary = pd.Series({
        "observations": _cov["observations"],
        "violations": _cov["violations"],
        "expected_violations_at_5pct": _cov["expected"],
        "empirical_violation_rate": _cov["violation_rate"],
        "kupiec_POF_LR": _cov["kupiec_LR"],
        "kupiec_POF_pvalue": _cov["kupiec_p"],
        "christoffersen_indep_LR": _cov["indep_LR"],
        "christoffersen_indep_pvalue": _cov["indep_p"],
        "christoffersen_cc_LR": _cov["cc_LR"],
        "christoffersen_cc_pvalue": _cov["cc_p"],
        "mean_VaR_95": float(bt["VaR_95"].mean()),
        "mean_ES_95": float(bt["ES_95"].mean()),
        "acerbi_szekely_Z2": es_test2["statistic"],
        "acerbi_szekely_Z2_pvalue": es_test2["p_value"],
        "acerbi_szekely_Z1": es_test1["statistic"],
        "acerbi_szekely_Z1_pvalue": es_test1["p_value"],
    }, name="walk_forward_VaR_ES_backtest")
    print("\nVaR/ES backtest (95%, equal-weight equity, one-week horizon):")
    display(summary.to_frame("value"))
    print(
        "Kupiec p tests unconditional coverage (H0: violation rate = 5%); Christoffersen indep p "
        "tests no clustering of breaches; the JOINT conditional-coverage p (chi2(2), LR_cc = LR_uc "
        "+ LR_ind) tests both at once -- a VaR can pass each separately and still fail the joint "
        "null. p-values above 0.05 do not reject."
    )
    print(
        f"Acerbi-Szekely ES test: Z2 = {es_test2['statistic']:.4f} (p = {es_test2['p_value']:.3f}). "
        "E[Z] = 0 under a correct predictive distribution. Z < 0 means realized tail losses exceeded "
        "the model's own ES (ES understated -- the direction that matters); Z > 0 means the ES was "
        "conservative. The null is simulated by resampling each date's own predictive scenario set, "
        "so no distributional assumption is imposed. Z1 conditions on the breaches (severity GIVEN a "
        "breach) and is undefined when there are none; Z2 is defined either way."
    )

    # ---- Coverage by volatility regime ---------------------------------------------------------
    # The long sample's real payoff: a VaR that covers on average but fails in stress is a broken
    # VaR, and a recent-calm-window backtest can never see it. Tercile cut-points come from the
    # realized panel -- a DIAGNOSTIC partition, declared: it cannot flatter the VaR, because every
    # VaR was formed at its own as-of date without them.
    if bt["trailing_vol_21d"].notna().sum() >= 15:
        _tv = bt["trailing_vol_21d"]
        _c1, _c2 = float(_tv.quantile(1 / 3)), float(_tv.quantile(2 / 3))
        bt["vol_regime"] = pd.cut(_tv, [-np.inf, _c1, _c2, np.inf],
                                  labels=["calm", "normal", "stressed"])
        _rows = []
        for _lab in ["calm", "normal", "stressed"]:
            _m = (bt["vol_regime"] == _lab).to_numpy()
            if _m.sum() == 0:
                continue
            _c = coverage_summary(bt.loc[_m, "violation"].to_numpy(), 0.05)
            _row = {"regime": _lab, "n": _c["observations"], "violations": _c["violations"],
                    "expected": _c["expected"], "violation_rate": _c["violation_rate"],
                    "kupiec_p": _c["kupiec_p"],
                    "mean_trailing_vol": float(bt.loc[_m, "trailing_vol_21d"].mean()),
                    "mean_VaR_95": float(bt.loc[_m, "VaR_95"].mean()),
                    "mean_ES_95": float(bt.loc[_m, "ES_95"].mean())}
            if _m.sum() >= 20 and bt_scenario_matrix is not None:
                _z = acerbi_szekely_es_test(bt.loc[_m, "realized_1w_return"].to_numpy(),
                                            bt.loc[_m, "VaR_95"].to_numpy(),
                                            bt.loc[_m, "ES_95"].to_numpy(),
                                            bt_scenario_matrix[_m], alpha=0.05,
                                            n_sim=ES_BACKTEST_N_SIM,
                                            rng=np.random.default_rng(RNG_SEED + 991), which=2)
                _row["ES_Z2"] = _z["statistic"]
                _row["ES_Z2_p"] = _z["p_value"]
            _rows.append(_row)
        regime_coverage = pd.DataFrame(_rows).set_index("regime")
        print(f"\nCoverage by trailing-volatility regime (terciles at {_c1:.1%} / {_c2:.1%} "
              "annualized 21-day realized vol, each known at its as-of date):")
        display(regime_coverage.round(4))
        print("A well-calibrated engine violates at ~5% in EVERY regime. Systematically too few "
              "breaches in 'calm' and too many in 'stressed' is the classic unconditional-model "
              "signature -- exactly what the conditional-volatility (FHS) filter exists to remove.")

    # ---- Coverage by calendar year (long backtests only) ----------------------------------------
    _years = bt["asof"].dt.year
    if _years.nunique() >= 3:
        _rows = []
        for _y, _g in bt.groupby(_years):
            _c = coverage_summary(_g["violation"].to_numpy(), 0.05)
            _rows.append({"year": int(_y), "n": _c["observations"], "violations": _c["violations"],
                          "expected": round(_c["expected"], 2), "violation_rate": _c["violation_rate"],
                          "mean_VaR_95": float(_g["VaR_95"].mean()),
                          "worst_1w_loss": float(-_g["realized_1w_return"].min())})
        year_coverage = pd.DataFrame(_rows).set_index("year")
        print("\nCoverage by calendar year (the regimes the long sample bought: crises are visible here):")
        display(year_coverage.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(bt["pit"], bins=10, range=(0, 1), edgecolor="white", alpha=0.85)
    axes[0].axhline(len(bt) / 10.0, color="black", lw=1, ls="--", label="uniform")
    axes[0].set_title("PIT / rank histogram (should be ~uniform)")
    axes[0].set_xlabel("PIT of realized return")
    axes[0].legend()
    axes[1].plot(bt["asof"], bt["VaR_95"], label="VaR 95% (loss)", lw=1.3)
    axes[1].plot(bt["asof"], bt["ES_95"], label="ES 95% (loss)", lw=1.0, ls="--", alpha=0.8)
    axes[1].plot(bt["asof"], -bt["realized_1w_return"], label="realized loss", lw=0.9, alpha=0.8)
    viol = bt[bt["violation"] == 1]
    axes[1].scatter(viol["asof"], -viol["realized_1w_return"], color="red", s=25, zorder=5, label="violation")
    axes[1].set_title("VaR / ES vs realized loss over time")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Too few usable observations for a meaningful backtest; increase BACKTEST_N_DATES "
          "or widen the data window.")



## 4. Horizon repricing

Scenarios are mapped back into instrument values via:

- equity repricing from simulated one-week log returns,
- scenario-wise repricing of a synthetic semi-annual coupon Treasury note from the simulated key-rate curve.


In [ ]:
# Treasury note definition and current repricing
issue_date = pd.Timestamp(effective_date)
horizon_date = issue_date + BDay(HORIZON_BDAYS)
maturity_date = issue_date + pd.DateOffset(years=10)

coupon_schedule = generate_coupon_schedule(issue_date, maturity_date, freq=PAY_FREQ)

# Issue #8: the bond is priced off the full simulated curve. The tenor grid is taken from the
# active key-rate columns (ascending order), so the pricer is consistent with whatever tenors the
# rates block uses.
RATE_TENOR_YEARS = [TENOR_YEARS[t] for t in key_rates.columns]

current_bond_price = price_coupon_bond_from_curve(
    eval_date=issue_date,
    payment_dates=coupon_schedule,
    key_rate_series=current_key_rates,
    tenor_years=RATE_TENOR_YEARS,
    coupon_rate=COUPON_RATE,
    face_value=FACE_VALUE,
    freq=PAY_FREQ,
)

# Issue #10: clean/dirty decomposition. The DCF above returns the dirty (full) PV; accrued interest
# separates the clean price. At issue the note has just been struck, so accrued ~ 0.
accrued_at_issue = accrued_interest(issue_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)
accrued_at_horizon = accrued_interest(horizon_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)

print(f"Synthetic Treasury note dirty price at issue: {current_bond_price:,.4f}")
print(f"  accrued at issue: {accrued_at_issue:,.4f}  ->  clean: {current_bond_price - accrued_at_issue:,.4f}")
print(f"Issue date   : {issue_date.date()}")
print(f"Horizon date : {pd.Timestamp(horizon_date).date()}")
print(f"Maturity date: {maturity_date.date()}")
print(
    "Pricing convention (declared approximation): FRED CMT par yields are linearly interpolated "
    "across tenor and treated as continuously-compounded zero rates. Corporate bonds add a flat "
    "option-adjusted spread (OAS) for their rating bucket on top of that curve. P&L below is "
    "dirty-to-dirty; accrued interest is reported only for the clean/dirty split."
)

bond_horizon_prices = key_rate_scenarios.apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
)
bond_horizon_prices.name = "UST_10Y_2pct"
if HORIZON_MODE == "1Y":
    # total-return: add the coupon cash paid over the year (immaterial / typically zero at 1W)
    _govt_coupon_income = coupons_paid_between(issue_date, horizon_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)
    bond_horizon_prices = bond_horizon_prices + _govt_coupon_income
bond_pnl = bond_horizon_prices - current_bond_price

print(f"\nMean dirty PV at horizon: {bond_horizon_prices.mean():,.4f} "
      f"(accrued at horizon: {accrued_at_horizon:,.4f})")
display(distribution_summary(bond_horizon_prices).to_frame("UST_10Y_2pct_horizon_value"))
display(loss_summary_from_pnl(bond_pnl).to_frame("UST_10Y_2pct_one_week_pnl"))

# --- Corporate bonds: discount at the simulated Treasury curve + the simulated OAS for the rating.
# This is where the equity-credit link bites: scenarios with falling equities tend to carry widening
# spreads (joint bootstrap), so a corporate bond loses on both the rate and the credit channel.
corp_coupon_schedules = {}
corp_current_prices = {}
corp_horizon_prices = {}
corp_horizon_ratings = {}

# Credit migration setup (1Y only). The SYSTEMATIC credit factor M per scenario is the standardized
# credit state, oriented so a spread blow-out is a LOW M: every name's migration latent shares it, so
# downgrades/defaults cluster exactly in the scenarios where the simulated spreads widened. Normal
# scores (rank -> Phi^{-1}) keep M ~ N(0,1) so the marginal horizon-rating distribution stays faithful
# to the transition matrix while the loading injects the wrong-way dependence.
if CREDIT_MIGRATION_ENABLED:
    transition_matrix = standard_1y_transition_matrix(tuple(r for r in TRANSITION_RATINGS if r != "D"))
    migration_thresholds = build_migration_thresholds(transition_matrix)
    _avg_spread_chg = (credit_spread_scenarios - bucket_anchor_spreads).mean(axis=1)
    _ranks = _avg_spread_chg.rank() / (len(_avg_spread_chg) + 1.0)
    systematic_credit_M = -norm.ppf(_ranks.to_numpy())          # widening -> low M
    _mig_rng = np.random.default_rng(RNG_SEED + 777)

print("\nCorporate bonds (priced on Treasury-zero + rating OAS"
      + ("; 1Y rating migration / default ON):" if CREDIT_MIGRATION_ENABLED else "):"))
for spec in CORP_BONDS:
    name, rating = spec["name"], spec["rating"]
    cpn, mat_y = spec["coupon"], spec["maturity_years"]
    sched_c = generate_coupon_schedule(issue_date, issue_date + pd.DateOffset(years=mat_y), freq=PAY_FREQ)
    corp_coupon_schedules[name] = sched_c

    cur_spread = float(bucket_anchor_spreads[rating])
    corp_current_prices[name] = price_coupon_bond_from_curve(
        eval_date=issue_date, payment_dates=sched_c, key_rate_series=current_key_rates,
        tenor_years=RATE_TENOR_YEARS, coupon_rate=cpn, face_value=FACE_VALUE, freq=PAY_FREQ,
        spread=cur_spread,
    )
    if CREDIT_MIGRATION_ENABLED:
        # 1Y: draw each scenario's horizon rating (incl. default), reprice on the migrated rating's
        # simulated OAS (or recovery on default), and add the coupon cash paid over the year so the
        # horizon value is a total-return value.
        h_ratings, _defmask = simulate_horizon_ratings(rating, systematic_credit_M,
                                                       MIGRATION_FACTOR_LOADING, migration_thresholds, _mig_rng)
        corp_horizon_ratings[name] = h_ratings
        repriced = reprice_corporate_bond_with_migration(
            eval_date=horizon_date, payment_dates=sched_c, key_rate_scenarios=key_rate_scenarios,
            tenor_years=RATE_TENOR_YEARS, all_spread_scenarios=credit_spread_scenarios,
            horizon_ratings=h_ratings, coupon_rate=cpn, recovery_rate=RECOVERY_RATE,
            face_value=FACE_VALUE, freq=PAY_FREQ,
        )
        cpn_income = coupons_paid_between(issue_date, horizon_date, sched_c, cpn, FACE_VALUE, PAY_FREQ)
        corp_horizon_prices[name] = repriced + cpn_income
        pnl_c = corp_horizon_prices[name] - corp_current_prices[name]
        dflt = float(_defmask.mean())
        downg = float(np.mean(np.array([TRANSITION_RATINGS.index(r) for r in h_ratings]) > TRANSITION_RATINGS.index(rating)))
        print(f"  {name} ({rating}, {cpn:.2%} {mat_y}Y): current OAS {cur_spread*1e4:,.0f}bps, "
              f"price {corp_current_prices[name]:,.3f} -> mean horizon {corp_horizon_prices[name].mean():,.3f} "
              f"(coupon +{cpn_income:.2f}; P(default) {dflt:.2%}, P(downgrade) {downg:.1%}; mean 1Y P&L {pnl_c.mean():,.4f})")
    else:
        corp_horizon_prices[name] = reprice_corporate_bond_scenarios(
            eval_date=horizon_date, payment_dates=sched_c, key_rate_scenarios=key_rate_scenarios,
            tenor_years=RATE_TENOR_YEARS, spread_scenarios=credit_spread_scenarios[rating],
            coupon_rate=cpn, face_value=FACE_VALUE, freq=PAY_FREQ,
        )
        pnl_c = corp_horizon_prices[name] - corp_current_prices[name]
        print(f"  {name} ({rating}, {cpn:.2%} {mat_y}Y): current OAS {cur_spread*1e4:,.0f}bps, "
              f"price {corp_current_prices[name]:,.3f} -> mean horizon {corp_horizon_prices[name].mean():,.3f} "
              f"(mean 1w P&L {pnl_c.mean():,.4f})")

# --- FX forward(s): a long-foreign forward struck at-the-money on the valuation curve, so its MTM
# is zero at inception and the one-week P&L is pure FX (the simulated spot) plus USD discounting (the
# simulated Treasury curve). It is repriced on the SAME simulated spot and curve rows, so an FX move
# and a rate move hit it jointly. For the capital-normalized portfolio layer the forward is treated
# as a fully-funded position: its capital base (current_value) is the USD notional and its horizon
# value is USD notional + MTM, so the instrument return is MTM / USD-notional.
fx_forward_specs = {}
fx_forward_current_values = {}
fx_forward_horizon_values = {}

print("\nFX forwards (long-foreign, struck at-the-money on the valuation curve):")
for spec in FX_FORWARDS:
    name, pair = spec["name"], spec["pair"]
    # At 1Y the original short (3M) tenor would expire long before the one-year horizon, so anchor the
    # maturity PAST the horizon -- the forward keeps its original tenor outstanding AT the horizon (a
    # 3M forward as of the horizon date), mirroring the 1W setup. The USD leg is fully simulated
    # (usd_key_rate_scenarios); the foreign funding rate is held constant, an approximation that is
    # immaterial at 1W but a real simplification at 1Y (a simulated foreign curve would be the fix).
    if HORIZON_MODE == "1Y":
        fwd_maturity = horizon_date + pd.DateOffset(months=int(round(spec["maturity_years"] * 12)))
    else:
        fwd_maturity = issue_date + pd.DateOffset(months=int(round(spec["maturity_years"] * 12)))
    spot0 = float(current_fx[pair])
    r_for = float(spec["foreign_funding_rate"])
    notl = float(spec["notional_foreign"])
    strike = atm_forward_fx_strike(
        eval_date=issue_date, maturity_date=fwd_maturity, spot=spot0,
        usd_key_rate_series=current_key_rates, usd_tenor_years=RATE_TENOR_YEARS,
        foreign_funding_rate=r_for,
    )
    usd_notional = notl * spot0
    mtm_h = reprice_fx_forward_scenarios(
        eval_date=horizon_date, maturity_date=fwd_maturity, spot_scenarios=fx_spot_scenarios[pair],
        usd_key_rate_scenarios=key_rate_scenarios, usd_tenor_years=RATE_TENOR_YEARS,
        strike=strike, foreign_funding_rate=r_for, notional_foreign=notl,
    )
    fx_forward_specs[name] = {
        "pair": pair, "maturity_date": fwd_maturity, "strike": strike,
        "foreign_funding_rate": r_for, "notional_foreign": notl, "usd_notional": usd_notional,
    }
    fx_forward_current_values[name] = usd_notional            # capital base (MTM at inception ~ 0)
    fx_forward_horizon_values[name] = usd_notional + mtm_h    # so return = MTM / USD-notional
    print(f"  {name} ({pair}, {spec['maturity_years']:.2f}Y): spot {spot0:.4f}, ATM strike {strike:.4f}, "
          f"USD notional {usd_notional:,.2f} -> mean horizon MTM {mtm_h.mean():,.4f} "
          f"(mean 1w return {(mtm_h.mean()/usd_notional):.4%})")

# Assemble the full instrument set: equities + Treasury + corporates + FX forward(s)
instrument_horizon_values = stock_price_scenarios.copy()
instrument_horizon_values["UST_10Y_2pct"] = bond_horizon_prices
for name in corp_horizon_prices:
    instrument_horizon_values[name] = corp_horizon_prices[name].to_numpy()
for name in fx_forward_horizon_values:
    instrument_horizon_values[name] = fx_forward_horizon_values[name].to_numpy()

current_values = current_prices.copy()
current_values["UST_10Y_2pct"] = current_bond_price
for name in corp_current_prices:
    current_values[name] = corp_current_prices[name]
for name in fx_forward_current_values:
    current_values[name] = fx_forward_current_values[name]

instrument_pnl = instrument_horizon_values.sub(current_values, axis=1)
instrument_returns = instrument_pnl.div(current_values, axis=1)

print("\nHorizon values:")
display(instrument_horizon_values.describe().T)

print("Instrument-level one-week P&L:")
display(instrument_pnl.describe().T)

print("Capital-normalized one-week return/loss metrics:")
display(pd.DataFrame({
    col: loss_summary_from_pnl(instrument_returns[col], alpha=CVAR_ALPHA)
    for col in instrument_returns.columns
}).T)


In [ ]:

# Selected instrument-level visualizations
plot_histogram(instrument_horizon_values[REPRESENTATIVE_EQUITY], f"Horizon-value distribution — {REPRESENTATIVE_EQUITY}", "Horizon value")
plot_histogram(instrument_pnl[REPRESENTATIVE_EQUITY], f"P&L distribution — {REPRESENTATIVE_EQUITY}", "One-week P&L")
plot_histogram(instrument_horizon_values["UST_10Y_2pct"], "Horizon-value distribution — UST_10Y_2pct", "Horizon value")
plot_histogram(instrument_pnl["UST_10Y_2pct"], "P&L distribution — UST_10Y_2pct", "One-week P&L")



## 5. Portfolio layer under capital-normalized scenario returns

Portfolio comparisons are performed on **one-week simple returns**, not raw dollar P&L per instrument unit. This keeps tail-risk comparisons economically interpretable when assets have different nominal prices.


In [ ]:
# Candidate portfolio weights in capital-normalized return space
all_assets = instrument_returns.columns.tolist()
equity_assets = TICKERS.copy()
corp_assets = [spec["name"] for spec in CORP_BONDS]
fx_assets = [spec["name"] for spec in FX_FORWARDS]

equal_weight = pd.Series(1.0 / len(all_assets), index=all_assets)

equity_only = pd.Series(0.0, index=all_assets)
equity_only.loc[equity_assets] = 1.0 / len(equity_assets)

# Treasury-heavy: 60% Treasury, 40% spread across equities, no corporate credit (sums to 1 over the
# enlarged universe — the previous fixed-denominator version no longer summed to one once corporate
# bonds were added).
bond_heavy = pd.Series(0.0, index=all_assets)
bond_heavy.loc[equity_assets] = 0.40 / len(equity_assets)
bond_heavy["UST_10Y_2pct"] = 0.60

# Credit-tilt: deliberately loads the corporate (credit-spread) names to expose the new risk block.
credit_tilt = pd.Series(0.0, index=all_assets)
credit_tilt.loc[equity_assets] = 0.40 / len(equity_assets)
credit_tilt["UST_10Y_2pct"] = 0.20
if corp_assets:
    for c in corp_assets:
        credit_tilt[c] = 0.40 / len(corp_assets)

# FX-overlay: equities + Treasury with a sleeve allocated to the FX forward(s), to exercise the FX
# block in the portfolio. The forward's capital base is its USD notional, so a weight here is the
# fraction of capital carried as the (fully-funded) FX position. Sums to one over the universe.
fx_overlay = pd.Series(0.0, index=all_assets)
fx_overlay.loc[equity_assets] = 0.40 / len(equity_assets)
fx_overlay["UST_10Y_2pct"] = 0.40
if fx_assets:
    for f in fx_assets:
        fx_overlay[f] = 0.20 / len(fx_assets)

inverse_vol = inverse_vol_weights_from_scenarios(instrument_returns)

# Issue #3: exact long-only min-CVaR via the Rockafellar-Uryasev LP (no more random search).
min_cvar_long_only, min_cvar_info = search_long_only_min_cvar_portfolio(
    instrument_returns=instrument_returns,
    alpha=CVAR_ALPHA,
    max_weight=CVAR_MAX_WEIGHT,
)

portfolio_weights = {
    "equal_weight": equal_weight,
    "equity_only": equity_only,
    "bond_heavy": bond_heavy,
    "credit_tilt": credit_tilt,
    "fx_overlay": fx_overlay,
    "inverse_vol": inverse_vol,
    "min_cvar_long_only": min_cvar_long_only,
}

portfolio_returns = {}
portfolio_metrics = {}
for name, w in portfolio_weights.items():
    port_ret, metrics = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    portfolio_returns[name] = port_ret
    portfolio_metrics[name] = metrics

print("Long-only min-CVaR LP diagnostics:")
display(min_cvar_info.to_frame("value"))

print("Portfolio weights:")
display(pd.DataFrame(portfolio_weights).T)

print("Portfolio one-week return/loss metrics:")
display(pd.DataFrame(portfolio_metrics).T.sort_values("ES_95_loss"))


In [ ]:
# Out-of-sample (fresh-draw) portfolio tail risk (issue #4)
# The min-CVaR weights are chosen to minimize ES on the in-sample scenario set, so the in-sample
# ES is optimistically biased (selection on noise). We simulate an INDEPENDENT scenario set with a
# different seed, reprice every instrument (equities, Treasury, corporates, FX forward), and
# re-evaluate the SAME weights to get an honest ES.

sim_oos = simulate_joint_horizon(
    var_res=var_res,
    resid_hist=var_res.resid,
    last_state=drivers.iloc[-1].to_numpy(),
    equity_block=equity_block,
    rates_block=rates_block,
    current_prices=current_prices,
    current_key_rates=current_key_rates,
    horizon_bdays=HORIZON_BDAYS,
    n_scenarios=N_SCENARIOS,
    avg_block_length=effective_block_length,
    rng_seed=VALIDATION_SEED,
    daily_rf=current_daily_rf,
    resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
    credit_block=credit_block,
    current_spreads=current_spreads,
    fx_block=fx_block,
    current_fx=current_fx,
    dynamics=DRIVER_DYNAMICS,
    demeaned_drivers=demeaned_drivers,
    driver_mean=driver_mean,
    crisis_mode=CRISIS_MODE,
    crisis_prob=crisis_prob_eff,
    crisis_scale=crisis_scale_eff,
    crisis_pool=crisis_pool_eff,
    copula=t_copula_model,
    asset_copula=asset_copula_model,
    fhs_filter=fhs_filter,
    ou_rates=ou_rates, ou_credit=ou_credit,
    credit_bridge=credit_bridge,
)

key_rate_scenarios_oos = sim_oos["key_rate_scenarios"]
credit_spread_scenarios_oos = sim_oos["credit_spread_scenarios"]
fx_spot_scenarios_oos = sim_oos["fx_spot_scenarios"]

bond_horizon_prices_oos = key_rate_scenarios_oos.apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
)

instrument_horizon_values_oos = sim_oos["stock_price_scenarios"].copy()
instrument_horizon_values_oos["UST_10Y_2pct"] = bond_horizon_prices_oos
for spec in CORP_BONDS:
    name, rating = spec["name"], spec["rating"]
    instrument_horizon_values_oos[name] = reprice_corporate_bond_scenarios(
        eval_date=horizon_date, payment_dates=corp_coupon_schedules[name],
        key_rate_scenarios=key_rate_scenarios_oos, tenor_years=RATE_TENOR_YEARS,
        spread_scenarios=credit_spread_scenarios_oos[rating],
        coupon_rate=spec["coupon"], face_value=FACE_VALUE, freq=PAY_FREQ,
    ).to_numpy()

# FX forward(s) on the independent draw: same struck strike / maturity / notional as the in-sample
# leg (from fx_forward_specs), repriced on the fresh spot and curve, then expressed as USD notional
# + MTM so it matches the fully-funded current_value used in-sample.
for name, fwd in fx_forward_specs.items():
    mtm_oos = reprice_fx_forward_scenarios(
        eval_date=horizon_date, maturity_date=fwd["maturity_date"],
        spot_scenarios=fx_spot_scenarios_oos[fwd["pair"]], usd_key_rate_scenarios=key_rate_scenarios_oos,
        usd_tenor_years=RATE_TENOR_YEARS, strike=fwd["strike"],
        foreign_funding_rate=fwd["foreign_funding_rate"], notional_foreign=fwd["notional_foreign"],
    )
    instrument_horizon_values_oos[name] = (fwd["usd_notional"] + mtm_oos).to_numpy()

instrument_returns_oos = instrument_horizon_values_oos.sub(current_values, axis=1).div(current_values, axis=1)

oos_rows = {}
for name, w in portfolio_weights.items():
    _, m_in = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    _, m_oos = portfolio_metrics_from_instrument_returns(w, instrument_returns_oos, alpha=CVAR_ALPHA)
    oos_rows[name] = {
        "VaR_95_in_sample": m_in["VaR_95_loss"],
        "VaR_95_out_of_sample": m_oos["VaR_95_loss"],
        "ES_95_in_sample": m_in["ES_95_loss"],
        "ES_95_out_of_sample": m_oos["ES_95_loss"],
    }

oos_table = pd.DataFrame(oos_rows).T
oos_table["ES_in_sample_optimism"] = oos_table["ES_95_out_of_sample"] - oos_table["ES_95_in_sample"]
print("In-sample vs out-of-sample one-week tail risk (independent scenario draw):")
display(oos_table.sort_values("ES_95_out_of_sample"))
print(
    "A positive 'ES_in_sample_optimism' for min_cvar_long_only is the selection-on-noise effect: "
    "the weights were fit to the in-sample tail, so their ES degrades out-of-sample. Policy "
    "portfolios (equal_weight, equity_only, bond_heavy, credit_tilt, fx_overlay) are not fit to the "
    "scenarios and should show only sampling noise between the two columns."
)


In [ ]:

# Portfolio-level distributions and weighted tail contributions
for name, port_ret in portfolio_returns.items():
    plot_histogram(port_ret, f"Portfolio one-week return distribution — {name}", "One-week portfolio return")

tail_table = tail_contribution_table(
    portfolio_return=portfolio_returns["min_cvar_long_only"],
    instrument_returns=instrument_returns,
    weights=portfolio_weights["min_cvar_long_only"],
    alpha=CVAR_ALPHA,
)

display(tail_table)


## Limitations and research extensions

This is a research engine, not a production risk system or a risk-neutral pricing library. Key limitations are deliberately explicit: Treasury CMT inputs are used as a simplified zero-curve representation; the long-history Moody's two-point credit driver is bridged into current rating-bucket OAS sensitivities for repricing; the EUR funding curve is simplified in the FX-forward block; and the 1Y branch is materially more model-dependent than the 1W mainline.

The Entropy Pooling / Flexible Probabilities research branch is intentionally **not** part of this release. It is maintained as a separate conditioning layer over the physical scenario distribution.
